# Exercise 05: Classification Trees, Random Forest and Gradient Boosting – Heart Disease

**Dataset:** 918 patients from five hospitals (Cleveland, Hungary, Switzerland, Long Beach VA, Stalog), combined by
fedesoriano (2021) from the UCI heart disease data sets. Source:
[Kaggle – Heart Failure Prediction](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction).

| variable | meaning |
|---|---|
| `Age`, `Sex` | age (years), sex (M/F) |
| `ChestPainType` | TA typical angina, ATA atypical angina, NAP non-anginal pain, ASY asymptomatic |
| `RestingBP` | resting blood pressure (mm Hg) |
| `Cholesterol` | serum cholesterol (mg/dl) |
| `FastingBS` | fasting blood sugar > 120 mg/dl (1) or not (0) |
| `RestingECG` | resting ECG result (Normal, ST, LVH) |
| `MaxHR` | maximum heart rate achieved in the exercise test |
| `ExerciseAngina` | exercise-induced angina (Y/N) |
| `Oldpeak` | ST depression in the ECG under exercise |
| `ST_Slope` | slope of the ST segment under exercise (Up, Flat, Down) |
| `HeartDisease` | 1 = coronary heart disease diagnosed, 0 = normal – **target** |

**Business question:** Can routine examinations help to identify patients with heart disease? In a medical setting,
**missing a sick patient** (false negative) is much worse than a false alarm that leads to further tests.

**Slides:** Part 04 «Classification models» – classification trees (anatomy, Gini, growing, predicting), random forest &
gradient boosting for classification, evaluation; slide «Putting it together in scikit-learn».

**After this exercise you can ...**
- compute Gini impurity and the impurity decrease of a split by hand,
- fit, visualise, interpret and prune a classification tree,
- tune random forest and gradient boosting classifiers with `GridSearchCV`,
- compare classifiers with ROC/AUC and choose a threshold for a required recall.

## Concept

**Gini impurity** of a node with class shares $\hat p_k$:

$$G = 1 - \sum_{k=1}^{K} \hat p_k^2 \qquad (0 = \text{pure node},\; 0.5 = \text{maximally mixed for two classes})$$

**Impurity decrease** of a split into a left node $L$ and a right node $R$ (with $n_L + n_R = n$ observations):

$$\Delta G = G_{\text{parent}} - \Big(\frac{n_L}{n}\,G_L + \frac{n_R}{n}\,G_R\Big)$$

The tree chooses greedily the (feature, threshold) with the largest $\Delta G$ and repeats recursively until a stopping
rule applies (`max_depth`, `min_samples_leaf`, pure node). Entropy is an alternative impurity measure with almost the same
shape. Large trees can be pruned with `ccp_alpha`.

**Prediction.** A leaf predicts the class shares of its training observations, e.g. 145 sick : 125 healthy →
$\hat p(\text{sick}) = 145/270 \approx 0.54$ → predict "sick" (if $\tau = 0.5$). Trees output **probabilities**, which is
the basis for thresholds and ROC curves.

**Ensembles for classification.**
- *Random forest:* many deep trees on bootstrap samples with random feature subsets (default $m = \sqrt p$); the forest
  averages the class probabilities of the trees.
- *Gradient boosting:* many small trees, built one after the other; each tree fits the pseudo-residuals (negative
  gradient) of the log-loss.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score, precision_score,
                             roc_auc_score, roc_curve, classification_report)
from sklearn.inspection import permutation_importance

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Gini impurity by hand

A node contains 200 patients, 110 with heart disease and 90 without. Two candidate splits are evaluated:

| candidate split | left node (yes) | right node (no) |
|---|---|---|
| `ST_Slope = Flat?` | 100 patients: 80 sick / 20 healthy | 100 patients: 30 sick / 70 healthy |
| `ExerciseAngina = Y?` | 80 patients: 64 sick / 16 healthy | 120 patients: 46 sick / 74 healthy |

a) Compute the Gini impurity of the parent node.

b) Compute $G_L$, $G_R$, the weighted impurity and $\Delta G$ for both candidate splits. Which split does the tree choose?

c) What does the left leaf of the chosen split predict (class and probability)?

d) Verify your results with Python.

In [ ]:
def gini(n_sick, n_healthy):
    """Gini impurity of a node with two classes."""
    n = n_sick + n_healthy
    return 1 - ((n_sick / n)**2 + (n_healthy / n)**2)

g_parent = gini(110, 90)
print(f'Parent: G = {g_parent:.4f}')

splits = {'ST_Slope = Flat?':    ((80, 20), (30, 70)),
          'ExerciseAngina = Y?': ((64, 16), (46, 74))}
for name, (left, right) in splits.items():
    n_l, n_r = sum(left), sum(right)
    g_l, g_r = gini(*left), gini(*right)
    g_w = n_l / (n_l + n_r) * g_l + n_r / (n_l + n_r) * g_r
    print(f'{name:20s} G_L = {g_l:.4f}, G_R = {g_r:.4f}, weighted G = {g_w:.4f}, Delta G = {g_parent - g_w:.4f}')

**Solution:**

a) $G = 1-(0.55^2+0.45^2) = 1-(0.3025+0.2025) = 0.495$.

b)
| split | $G_L$ | $G_R$ | weighted $G$ | $\Delta G$ |
|---|---|---|---|---|
| `ST_Slope = Flat?` | $1-(0.8^2+0.2^2)=0.32$ | $1-(0.3^2+0.7^2)=0.42$ | $0.5\cdot0.32+0.5\cdot0.42=0.37$ | **0.125** |
| `ExerciseAngina = Y?` | $1-(0.8^2+0.2^2)=0.32$ | $1-(0.383^2+0.617^2)\approx0.473$ | $0.4\cdot0.32+0.6\cdot0.473\approx0.412$ | 0.083 |

`ST_Slope = Flat?` reduces the impurity more and is chosen. Note: both left nodes are equally pure (80 % sick) – the
difference comes from the right nodes and from the different node sizes.

c) The left leaf predicts "heart disease" with $\hat p = 80/100 = 0.8$.

## Exercise 2: Import, check and prepare the data

a) Import the data and check the descriptive statistics. Which values of `RestingBP` and `Cholesterol` are
implausible?

b) Remove the row with `RestingBP == 0`. For `Cholesterol == 0` (presumably not measured) create an indicator
`Cholesterol_missing` and replace the zeros by the median of the measured values.

c) Create dummy variables for the categorical features (`pd.get_dummies(..., drop_first=True, dtype=int)`) and split the
data into a stratified training (80 %) and test set (20 %) with `random_state=42`.

d) What is the share of patients with heart disease?

In [ ]:
# a) Import the data
df = pd.read_csv('./Data/heart.csv', sep=',')
print(df.shape)
print(df.describe().round(1).T)
print('RestingBP == 0:  ', (df['RestingBP'] == 0).sum())
print('Cholesterol == 0:', (df['Cholesterol'] == 0).sum())

# b) Implausible values
df = df[df['RestingBP'] > 0].copy()
df['Cholesterol_missing'] = (df['Cholesterol'] == 0).astype(int)
median_chol = df.loc[df['Cholesterol'] > 0, 'Cholesterol'].median()
df.loc[df['Cholesterol'] == 0, 'Cholesterol'] = median_chol
print('Median cholesterol used for imputation:', median_chol)

# c) Dummy variables and train/test split
X = pd.get_dummies(df.drop(columns='HeartDisease'), drop_first=True, dtype=int)
y = df['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print('Training set:', X_train.shape, ' Test set:', X_test.shape)
print(list(X.columns))

# d) Share of patients with heart disease
print('Share with heart disease:', round(y.mean(), 3))
print('Heart disease rate by Cholesterol_missing:')
print(df.groupby('Cholesterol_missing')['HeartDisease'].mean().round(3))

**Solution:**

a) A blood pressure of 0 mm Hg (1 patient) and a cholesterol of 0 mg/dl (172 patients, almost 19 %) are physiologically
impossible – these are missing values coded as 0.

b) Simply deleting 172 patients would waste a lot of data. Interestingly, the missing cholesterol values are **not random**:
patients without a cholesterol measurement have heart disease much more often (≈ 88 % vs. ≈ 48 %; presumably they
come mainly from one of the five hospitals, which did not record cholesterol). The
indicator variable lets the model use this information. (Strictly speaking, the median should be computed on the
training data only; the effect here is negligible.)

d) About 55 % of the patients have heart disease – the classes are roughly balanced.

## Exercise 3: A small classification tree

a) Fit a `DecisionTreeClassifier(max_depth=3, random_state=42)` on the training set and visualise it with
`tree.plot_tree(..., class_names=['healthy', 'sick'], filled=True)`.

b) Interpret the root split and the second level. Do the splits make medical sense?

c) Check the root split by hand: the code below prints the class counts of the root and its two children. Compute the
Gini values and $\Delta G$ and compare them with the plot.

d) Compute accuracy, recall and AUC on the test set.

In [ ]:
# a) Fit and plot the tree
clf_tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
plt.figure(figsize=(20, 8))
tree.plot_tree(clf_tree, feature_names=list(X_train.columns), class_names=['healthy', 'sick'],
               filled=True, rounded=True, fontsize=9, precision=2)
plt.show()

# c) Class counts of the root and its children
t = clf_tree.tree_
for node, name in [(0, 'root'), (t.children_left[0], 'left child'), (t.children_right[0], 'right child')]:
    counts = np.round(t.value[node][0] * t.n_node_samples[node]).astype(int)
    print(f'{name:12s}: n = {t.n_node_samples[node]}, healthy = {counts[0]}, sick = {counts[1]}, Gini = {t.impurity[node]:.3f}')

n0, nl, nr = t.n_node_samples[0], t.n_node_samples[t.children_left[0]], t.n_node_samples[t.children_right[0]]
delta = t.impurity[0] - (nl / n0 * t.impurity[t.children_left[0]] + nr / n0 * t.impurity[t.children_right[0]])
print(f'Delta G of the root split = {delta:.3f}')

# d) Test metrics
y_pred_tree = clf_tree.predict(X_test)
print(f'\nAccuracy = {accuracy_score(y_test, y_pred_tree):.3f}, Recall = {recall_score(y_test, y_pred_tree):.3f}, '
      f'AUC = {roc_auc_score(y_test, clf_tree.predict_proba(X_test)[:, 1]):.3f}')

**Solution:**

b) The root split is on **ST_Slope_Up** (upsloping ST segment in the exercise ECG): patients *without* an upsloping
ST segment (left, 346 of 417 sick) are mostly sick, patients with an upsloping segment (right, 257 of 316 healthy) are
mostly healthy – a well-known ECG criterion for ischaemia. On the second level the tree uses **sex** (men are at higher
risk) and **Cholesterol_missing**; further down `MaxHR` (a low maximum heart rate under exercise indicates disease),
`FastingBS` and `ExerciseAngina_Y`. Most splits are medically plausible. `Cholesterol_missing` is not a medical
characteristic but a property of the data collection (presumably a hospital that did not measure cholesterol and saw many
sick patients) – the model uses it because it works *in this data*, but it would not transfer to another hospital.

c) Root: $G = 1-ig((328/733)^2+(405/733)^2ig) pprox 0.494$; left: $1-ig((71/417)^2+(346/417)^2ig) pprox 0.283$;
right: $1-ig((257/316)^2+(59/316)^2ig) pprox 0.304$;
$\Delta G = 0.494 - (417/733\cdot 0.283 + 316/733\cdot 0.304) pprox 0.203$ – identical to the plot.

d) The small tree reaches an accuracy of ≈ 0.84, a recall of ≈ 0.83 and an AUC of ≈ 0.90 – remarkable for only 8 rules.

## Exercise 4: Overfitting and pruning

a) Fit a tree without depth limit. Compare training and test accuracy with the depth-3 tree. How many leaves does the
full tree have?

b) Choose `ccp_alpha` with `GridSearchCV` (stratified 5-fold CV, `scoring='roc_auc'`) from the values of
`cost_complexity_pruning_path(X_train, y_train).ccp_alphas`. How many leaves does the best tree have, and what is its test
AUC?

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# a) Full tree vs. depth-3 tree
full_tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
for name, m in [('depth 3', clf_tree), ('full tree', full_tree)]:
    print(f'{name:10s}: leaves = {m.get_n_leaves():3d}, train accuracy = {m.score(X_train, y_train):.3f}, '
          f'test accuracy = {m.score(X_test, y_test):.3f}, '
          f'test AUC = {roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]):.3f}')

# b) Pruning with GridSearchCV
alphas = full_tree.cost_complexity_pruning_path(X_train, y_train).ccp_alphas
grid_tree = GridSearchCV(DecisionTreeClassifier(random_state=42), {'ccp_alpha': alphas},
                         cv=cv, scoring='roc_auc')
grid_tree.fit(X_train, y_train)
pruned_tree = grid_tree.best_estimator_
print(f"\nBest ccp_alpha = {grid_tree.best_params_['ccp_alpha']:.4f}, CV AUC = {grid_tree.best_score_:.3f}")
print(f'Pruned tree: leaves = {pruned_tree.get_n_leaves()}, '
      f'test AUC = {roc_auc_score(y_test, pruned_tree.predict_proba(X_test)[:, 1]):.3f}')

**Solution:**

a) The full tree classifies the training data perfectly (training accuracy 1.0) with 108 leaves, but its test accuracy
(≈ 0.82) is *lower* than that of the depth-3 tree (≈ 0.84), and its AUC is much lower (≈ 0.81 vs. ≈ 0.90): in the full
tree every leaf is pure, so the predicted probabilities are only 0 or 1 – the ranking information is lost. Classic
overfitting.

b) The best pruned tree has only 11 leaves and a test AUC of ≈ 0.91, slightly better than the depth-3 tree. Pruning and
depth limits are two ways to control the variance of a tree.

## Exercise 5: Random forest and gradient boosting with GridSearchCV

Tune the two ensembles as on the slide «Putting it together in scikit-learn» (stratified 5-fold CV,
`scoring='roc_auc'`):

- `RandomForestClassifier(n_estimators=500, random_state=42)` with
  `{'max_features': ['sqrt', 0.5], 'min_samples_leaf': [1, 5, 20]}`
- `GradientBoostingClassifier(random_state=42)` with
  `{'learning_rate': [0.05, 0.1], 'n_estimators': [100, 300], 'max_depth': [2, 3]}`

Print the best parameters and the best CV AUC for both models. Which hyperparameters matter?

In [ ]:
# Random forest
grid_rf = GridSearchCV(RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
                       {'max_features': ['sqrt', 0.5], 'min_samples_leaf': [1, 5, 20]},
                       cv=cv, scoring='roc_auc')
grid_rf.fit(X_train, y_train)
print('Random forest - best parameters:', grid_rf.best_params_, ' CV AUC:', round(grid_rf.best_score_, 3))
print(pd.DataFrame(grid_rf.cv_results_)[['params', 'mean_test_score']].round(3).to_string(index=False))

# Gradient boosting
grid_gb = GridSearchCV(GradientBoostingClassifier(random_state=42),
                       {'learning_rate': [0.05, 0.1], 'n_estimators': [100, 300], 'max_depth': [2, 3]},
                       cv=cv, scoring='roc_auc')
grid_gb.fit(X_train, y_train)
print('\nGradient boosting - best parameters:', grid_gb.best_params_, ' CV AUC:', round(grid_gb.best_score_, 3))
print(pd.DataFrame(grid_gb.cv_results_)[['params', 'mean_test_score']].round(3).to_string(index=False))

**Solution:**

Both ensembles reach a CV AUC of about **0.93** (random forest: `max_features='sqrt'`, `min_samples_leaf=5`;
gradient boosting: `learning_rate=0.05`, `n_estimators=100`, `max_depth=3`). The differences between the parameter
combinations are small (0.915–0.931): random forests are robust to their hyperparameters (very large leaves,
`min_samples_leaf=20`, are slightly worse); for gradient boosting, more trees (300) are slightly *worse* than 100 – with
a small data set, boosting starts to overfit early. The large gain compared to a single tree (CV AUC ≈ 0.89) comes from
the ensemble itself, not from fine-tuning.

## Exercise 6: Model comparison on the test set

a) Add a logistic regression (with `StandardScaler`, in a `make_pipeline`) as a baseline. Plot the ROC curves of the
depth-3 tree, the pruned tree, the tuned random forest, the tuned gradient boosting and the logistic regression on the
test set, and print the AUCs.

b) For the best model: which threshold $\tau$ is needed to reach a **recall of at least 0.95**? What are the precision and
the false positive rate at this threshold? Compare with $\tau = 0.5$.

In [ ]:
models = {
    'Tree (depth 3)':      clf_tree,
    'Pruned tree':         pruned_tree,
    'Random forest':       grid_rf.best_estimator_,
    'Gradient boosting':   grid_gb.best_estimator_,
    'Logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train),
}

# a) ROC curves and AUC
plt.figure(figsize=(6.5, 6.5))
aucs = {}
for name, m in models.items():
    p = m.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, p)
    aucs[name] = roc_auc_score(y_test, p)
    plt.plot(fpr, tpr, linewidth=1.8, label=f'{name} (AUC = {aucs[name]:.3f})')
plt.plot([0, 1], [0, 1], color='grey', linestyle='--')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate (recall)')
plt.title('ROC curves on the test set')
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.show()
print(pd.Series(aucs).sort_values(ascending=False).round(3))

# b) Threshold for recall >= 0.95
best_name = max(aucs, key=aucs.get)
p_best = models[best_name].predict_proba(X_test)[:, 1]
fpr, tpr, thr = roc_curve(y_test, p_best)
i = np.argmax(tpr >= 0.95)                       # first (= highest) threshold with recall >= 0.95
tau_95 = thr[i]
print(f'\nBest model: {best_name}')
for tau in [0.5, tau_95]:
    y_hat = (p_best >= tau).astype(int)
    TN, FP, FN, TP = confusion_matrix(y_test, y_hat).ravel()
    print(f'tau = {tau:.3f}: recall = {TP/(TP+FN):.3f}, precision = {TP/(TP+FP):.3f}, '
          f'FPR = {FP/(FP+TN):.3f}, FN = {FN}, FP = {FP}')

**Solution:**

a) Gradient boosting (≈ 0.94), logistic regression (≈ 0.93) and random forest (≈ 0.93) are almost equally good on the
test set, the single trees are clearly worse (≈ 0.90–0.91). In this data set, a well-specified logistic regression is a strong competitor: the relationships are
largely monotonic and additive, so the extra flexibility of the ensembles does not pay off much. (With only 184 test
patients, AUC differences of ±0.01 are not meaningful.)

b) For gradient boosting, the threshold must be lowered to about **τ ≈ 0.3** to reach a recall of 0.95. The number of
missed sick patients drops from 12 to 5, while the number of false alarms rises from 11 to 17 (FPR from ≈ 0.13 to
≈ 0.21; precision from ≈ 0.89 to ≈ 0.85) – more healthy patients are sent for further examinations. In a screening context this trade-off is usually acceptable; the final decision is a medical one, not a
statistical one.

## Exercise 7: What drives the predictions?

Compute the **permutation importance** of the tuned random forest on the test set (`scoring='roc_auc'`, 20 repeats) and
plot the 10 most important features. Compare with the splits of the depth-3 tree.

In [ ]:
perm = permutation_importance(grid_rf.best_estimator_, X_test, y_test, scoring='roc_auc',
                              n_repeats=20, random_state=42, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values().tail(10)

plt.figure(figsize=(7, 4))
imp.plot(kind='barh', color='darkred')
plt.xlabel('Decrease in test AUC when the feature is shuffled')
plt.title('Permutation importance (random forest)')
plt.show()

**Solution:**

The most important features are `ST_Slope_Up`, `Cholesterol_missing`, `Sex_M` and `ExerciseAngina_Y`, followed by
the chest pain type, `FastingBS`, `MaxHR`, `ST_Slope_Flat` and `Oldpeak` – largely the same variables the small tree
uses for its first splits. Classical risk factors such as the measured cholesterol value or the resting blood pressure
contribute little. (Because several ST/exercise variables carry similar information, shuffling one of them hurts the
forest only moderately – correlated features share their importance.) Remember: importance describes what the *model*
uses for prediction, not causal effects – `Cholesterol_missing` is the best example.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')